# Figure 1 — GNN–Pearson Feature Graph (generator)

Generates the model feature graph: the 40-node economic network used by the
GNN–Pearson model.

- **GDP hub** (gold, center) connects unconditionally to all 39 predictor nodes.
- **Inner ring** (blue): 20 FRED macro indicators.
- **Outer ring** (green): 19 MRTS retail-trade series.
- **Edges** are drawn where the Pearson correlation |ρ| ≥ 0.7, coloured by type
  (red = FRED–FRED, green = MRTS–MRTS, purple = cross-ring).
- **Inset** (bottom-left): the per-node feature vector — 4 channels × 6-month window = 24 dims.

The data load mirrors `V72_Final.ipynb` exactly (same 39-series `SELECTED_FEATURES`,
same FRED/MRTS/GDP loaders). The correlation here is computed on the full 2006–2025
sample as a representative illustration; in the model it is re-estimated on each
walk-forward training fold (584–684 edges, mean ≈ 650, ~88% density).

In [ ]:
import pandas as pd, numpy as np, pathlib, itertools
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA_DIR = pathlib.Path("./data")
FRED_DIR = DATA_DIR / "input/new/ref_from_ST_FRED"
MRTS_DIR = DATA_DIR / "input/new/raw_from_MRTS"
GDP_CSV  = FRED_DIR / "Quarterly_GDP.csv"
OUTPUT_DIR = DATA_DIR / "output/new/GNN_corr/quarterly_walkfwd_v72_final"

FRED = ["F_PCE_Pers","F_PCES_Serv","F_AHETPI_Avg","F_CPIAUCSL_Cons","F_DSPI_Disp",
        "F_TOTALSL_Tota","F_Real_Pers","F_M2SL_M2","F_PCEDG_Dura","F_BUSLOANS_Comm",
        "F_Real_Disp","F_PPIACO_PPI","F_PAYEMS_Nonf","F_DGORDER_Dura","F_NASDAQCOM_NASD",
        "F_NEWORDER_Core","F_INDPRO_Indu","F_MANEMP_ISM","F_GS10_10Ye","F_WTISPLC_WTI"]
MRTS = ["MRTS_44Y72","MRTS_44W72","MRTS_44X72","MRTS_44Z72","MRTS_44000","MRTS_446",
        "MRTS_445","MRTS_722","MRTS_4451","MRTS_452","MRTS_454","MRTS_441","MRTS_444",
        "MRTS_44112","MRTS_448","MRTS_453","MRTS_447","MRTS_442","MRTS_4522"]

def load_fred():
    d=[]
    for f in sorted(FRED_DIR.glob("Monthly_*.csv")):
        df=pd.read_csv(f,parse_dates=["observation_date"]); vc=[c for c in df.columns if c!="observation_date"][0]
        df.rename(columns={"observation_date":"date"},inplace=True); df[vc]=pd.to_numeric(df[vc],errors="coerce")
        df=df.dropna(subset=[vc]).set_index("date")[[vc]]; p=f.stem.replace("Monthly_","").split("_")
        df.columns=[f"F_{p[0]}_{p[1][:4]}" if len(p)>=2 else f"F_{p[0]}"]; d.append(df)
    return pd.concat(d,axis=1).sort_index().resample("MS").last()

def load_mrts():
    d=[]
    for f in sorted(MRTS_DIR.glob("*.xlsx")):
        try:
            n=f.name.split("_")[0]; raw=pd.read_excel(f,header=None,skiprows=7)
            raw.columns=["Period","Value"]+[f"x{i}" for i in range(len(raw.columns)-2)]
            raw["Value"]=pd.to_numeric(raw["Value"],errors="coerce")
            raw["date"]=pd.to_datetime(raw["Period"].astype(str),format="%b-%Y",errors="coerce")
            raw=raw.dropna(subset=["date","Value"]).set_index("date")[["Value"]]; raw.columns=[f"MRTS_{n}"]; d.append(raw)
        except Exception: pass
    return pd.concat(d,axis=1).sort_index().resample("MS").last()

print("Loaders + 39-feature set defined.")

In [ ]:
fred_df = load_fred(); mrts_df = load_mrts()
gdp = pd.read_csv(GDP_CSV, parse_dates=["observation_date"]).rename(columns={"observation_date":"date"})
gdp["GDP"] = pd.to_numeric(gdp["GDP"], errors="coerce")
gdp = gdp.dropna(subset=["GDP"]).set_index("date")[["GDP"]]
idx = pd.date_range(min(fred_df.index.min(), mrts_df.index.min()),
                    max(fred_df.index.max(), mrts_df.index.max()), freq="MS")
merged = pd.concat([fred_df, mrts_df, gdp.shift(1).reindex(idx).ffill()], axis=1).sort_index()

fred = [c for c in FRED if c in merged.columns]
mrts = [c for c in MRTS if c in merged.columns]
cols = fred + mrts + ["GDP"]
C = merged[cols].loc["2006-01-01":"2025-12-01"].dropna().corr()
TAU = 0.7
nedges = sum(1 for a,b in itertools.combinations(fred+mrts,2) if abs(C.loc[a,b]) >= TAU)
print(f"Nodes: {len(cols)} ({len(fred)} FRED + {len(mrts)} MRTS + GDP)")
print(f"Non-hub edges at |rho| >= {TAU}: {nedges}  (density {nedges/741*100:.0f}% of 741 pairs)")

In [ ]:
pos = {"GDP": (0,0)}
for i,n in enumerate(fred): a=2*np.pi*i/len(fred);       pos[n]=(1.00*np.cos(a),1.00*np.sin(a))
for i,n in enumerate(mrts): a=2*np.pi*i/len(mrts)+0.1;   pos[n]=(1.95*np.cos(a),1.95*np.sin(a))

fig, ax = plt.subplots(figsize=(10,10))
for n in fred+mrts:                                    # GDP hub spokes
    ax.plot([0,pos[n][0]],[0,pos[n][1]], color="#D4AF37", alpha=0.10, lw=0.5, zorder=1)
for a,b in itertools.combinations(fred+mrts,2):        # predictor-predictor edges
    if abs(C.loc[a,b]) >= TAU:
        af,bf = a in fred, b in fred
        col = "#d62728" if (af and bf) else ("#2ca02c" if (not af and not bf) else "#9467bd")
        ax.plot([pos[a][0],pos[b][0]],[pos[a][1],pos[b][1]], color=col, alpha=0.22, lw=0.5, zorder=2)
ax.scatter([pos[n][0] for n in fred],[pos[n][1] for n in fred], s=210, c="#1f77b4", edgecolors="white", lw=1, zorder=3)
ax.scatter([pos[n][0] for n in mrts],[pos[n][1] for n in mrts], s=210, c="#2ca02c", edgecolors="white", lw=1, zorder=3)
ax.scatter([0],[0], s=900, c="#FFD700", edgecolors="black", lw=1.5, zorder=4)
ax.text(0,0,"GDP", ha="center", va="center", fontsize=10, fontweight="bold", zorder=5)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("GNN-Pearson Feature Graph  (40 nodes; edges |\u03c1| \u2265 0.7, training data)", fontsize=13)
leg=[Line2D([0],[0],marker='o',color='w',markerfacecolor='#FFD700',markeredgecolor='k',markersize=14,label='GDP hub (target)'),
     Line2D([0],[0],marker='o',color='w',markerfacecolor='#1f77b4',markersize=11,label='FRED macro (20)'),
     Line2D([0],[0],marker='o',color='w',markerfacecolor='#2ca02c',markersize=11,label='MRTS retail (19)'),
     Line2D([0],[0],color='#d62728',lw=2,label='FRED-FRED edge'),
     Line2D([0],[0],color='#2ca02c',lw=2,label='MRTS-MRTS edge'),
     Line2D([0],[0],color='#9467bd',lw=2,label='cross-ring edge'),
     Line2D([0],[0],color='#D4AF37',lw=2,alpha=0.5,label='GDP-hub spoke')]
ax.legend(handles=leg, loc="upper left", fontsize=9, framealpha=0.9)
iax=fig.add_axes([0.07,0.07,0.20,0.16]); iax.imshow(np.random.RandomState(0).randn(4,6), cmap="coolwarm", aspect="auto")
iax.set_yticks(range(4)); iax.set_yticklabels(["levels","velocity","volatility","accel."],fontsize=7)
iax.set_xticks([0,5]); iax.set_xticklabels(["t-5","t"],fontsize=7); iax.set_title("per-node X(t): 4x6 = 24 dims",fontsize=8)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "fig_1_feature_graph.png", dpi=130, bbox_inches="tight")
print("Saved:", OUTPUT_DIR / "fig_1_feature_graph.png")

## Notes

- Node count: **40** (20 FRED + 19 MRTS + GDP). Non-hub edges at |ρ| ≥ 0.7 on the
  full-sample window: **564** (~76% of the 741 possible predictor pairs); the per-fold
  graphs average ~650 edges (88%), the small difference reflecting the shorter,
  expanding training windows used per fold.
- The figure is saved to the model output directory alongside the Chapter-4 figures.
- Standard layout and colour scheme.